# BAG correlates explorer

Assembles, in one wide table:

- **Global BAG** (raw + Cole-corrected) for every session, from the current *production* `stacked` model's stored `predictions` rows (real out-of-fold values, not the full-data refit).
- **Regional BAG** (raw + Cole-corrected, one corrector per region), one column per `atlas__region`, predicted directly from the production artifact's own fitted per-region base estimators — no retraining. In-sample for sessions the model was trained on, out-of-sample for any added since. Raw is what actually feeds the meta-learner; corrected is what region-level analysis below uses.
- **Metadata**: SNBB + legacy demographics/session context (lab/protocol/study/group/scan_tag/scan_date, not just age/sex), questionnaire responses (SNBB only — hundreds of columns: PHQ9/GAD7/PCL-5/OASIS/BIG5/POMS/SCS/... plus lifestyle/environment/health fields), physiological measurements (hand grip, seca anthropometrics), and CAT12 QC (SIQR).

Then a generic `explore(feature)` helper: pass any metadata column name, get its correlation with global BAG plus a ranked bar chart of which brain regions correlate most strongly — numeric features use Pearson/Spearman, categorical features use group means + one-way ANOVA.

**Real data only — do not commit outputs.** `nbstripout` should already be wired via pre-commit; double check before pushing.

In [ ]:
import json

import cloudpickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from bagpipe.core.config import get_path
from bagpipe.db.base import get_engine
from bagpipe.models.tabular import build_region_matrix

# bagpipe's own tables (features, predictions, ...) live in the same SQLite
# file as brainlink's (participant, session, questionnaire_response, ...) —
# one engine covers both.
engine = get_engine()

## 1. Production model config — read live, not hardcoded

In [ ]:
reg_row = pd.read_sql(
    "select * from models_registry where name = 'stacked' and stage = 'production' "
    "order by trained_at desc limit 1",
    engine,
).iloc[0]
model_id = int(reg_row["model_id"])
model_config = json.loads(reg_row["config_json"])
print(f"production model_id={model_id} version={reg_row['version']} trained_at={reg_row['trained_at']}")
model_config

## 2. Regional BAG — from the production artifact, no retraining or fitting here

`predictions` only stores the global (meta-learner) BAG. But the saved artifact's
`region_estimators_` (fit once, at promotion time) can predict per-region age directly
on the current region matrix — no need to refit `RegionalStackingRegressor` here.

Two variants, both precomputed and stored on the artifact at promotion time
(`bagpipe.models.promote.promote` → `bagpipe.models.bias_correction.fit_region_correctors`):
**raw** (`region_estimators_` — untouched, what actually feeds the meta-learner) and
**Cole-corrected** (`region_correctors_`, one corrector per region). This notebook only
calls `.predict()`/`.transform()` on what's already fit — no correction is fit here.
Corrected is what region-level analysis (`explore()`, the brain-surface plots) should
use — raw regional predictions carry the same age-dependent bias the Cole correction
exists to remove.

In [ ]:
from pathlib import Path

from bagpipe.core.config import REPO_ROOT

feature_cfg = model_config.get("features", {})
datasets_dir = (
    REPO_ROOT / model_config["datasets_dir"] if model_config.get("datasets_dir") else get_path("datasets_dir")
)

X, y, groups, region_columns, session_ids = build_region_matrix(
    datasets_dir, metrics=feature_cfg.get("metrics"), atlases=feature_cfg.get("atlases")
)
print(f"{len(y)} sessions, {len(region_columns)} region/metric columns")

In [ ]:
with open(reg_row["artifact_path"], "rb") as f:
    production_model = cloudpickle.load(f)  # TIVSexAdjustedRegressor wrapping a fitted RegionalStackingRegressor

region_x, tiv, sex = X[:, :-2], X[:, -2], X[:, -1]
residuals = production_model.adjuster_.transform(region_x, tiv, sex)  # same TIV/sex adjustment used at training

stacker = production_model.model_
if not hasattr(stacker, "region_correctors_"):
    raise RuntimeError(
        f"model_id={model_id} version={reg_row['version']!r} predates per-region Cole correction "
        "(bagpipe.models.bias_correction.fit_region_correctors) — re-promote to get region_correctors_."
    )
region_names = stacker.region_names_
region_pred_age = np.column_stack(
    [
        stacker.region_estimators_[rname].predict(residuals[:, stacker.region_columns_[rname]])
        for rname in region_names
    ]
)
region_pred_age_corrected = np.column_stack(
    [stacker.region_correctors_[rname].transform(region_pred_age[:, i]) for i, rname in enumerate(region_names)]
)
# In-sample for sessions the model was trained on, genuinely out-of-sample for any session added
# to the cohort since promotion — not a leak-free CV estimate like the global BAG below.
regional_bag_raw = region_pred_age - y[:, None]
regional_bag_corrected = region_pred_age_corrected - y[:, None]

regional_bag_df = pd.DataFrame(regional_bag_raw, columns=[f"region_bag_raw__{r}" for r in region_names])
regional_bag_df.insert(0, "session_id", session_ids)
regional_bag_df.insert(0, "subject_key", groups)

regional_bag_corrected_df = pd.DataFrame(
    regional_bag_corrected, columns=[f"region_bag_corrected__{r}" for r in region_names]
)
regional_bag_corrected_df.insert(0, "session_id", session_ids)
regional_bag_corrected_df.insert(0, "subject_key", groups)

regional_bag_df = regional_bag_df.merge(regional_bag_corrected_df, on=["subject_key", "session_id"])
regional_bag_df.shape

## 3. Global BAG — real held-out predictions from `predictions`

In [ ]:
global_bag_df = pd.read_sql(
    "select subject_key, session_id, age_true, bag_raw as global_bag_raw, "
    "bag_corrected as global_bag_corrected from predictions where model_id = :mid",
    engine,
    params={"mid": model_id},
)
global_bag_df.shape

## 4. Metadata — everything we can get our hands on

One row per (subject_key, session_id). SNBB (`uid`/`session_id`) and legacy (12-digit id used as both) cohorts are unioned; questionnaire and physiological data only exist for SNBB.

In [ ]:
# SNBB demographics + session context — every non-path, non-PII column on session/demographics,
# not just age/sex, so protocol/study/group_label/scan_tag etc. are explorable too.
snbb_demo = pd.read_sql(
    """
    select s.uid as subject_key, s.session_id, s.subject_code, s.lab, s.protocol, s.study,
           s.group_label, s.scan_tag, s.scan_number, s.scan_date,
           d.age_at_scan as age, d.sex, d.weight_kg, d.height_m, d.dominant_hand
    from session s
    left join demographics d on d.session_id = s.session_id
    where s.uid is not null
    """,
    engine,
)

# Legacy demographics (bagpipe-owned table) — every column LegacyParticipant carries (PII columns
# from the source sheet were never ingested in the first place, see its docstring).
legacy_demo = pd.read_sql(
    """
    select subject_id as subject_key, subject_id as session_id, subject_id as subject_code,
           lab, study, group_label, scan_tag, scan_date, age_at_scan as age, gender as sex,
           weight, height
    from legacy_participant
    """,
    engine,
)

demo = pd.concat([snbb_demo, legacy_demo], ignore_index=True)
demo.shape

In [ ]:
# Questionnaire responses (SNBB only, keyed by subject_code): each row is one wide JSON blob,
# a subject can have several partial rows (different questionnaire_version passes) — merge them
# per subject, later (by ingested_at) values winning but not clobbering fields only present earlier.
qresp = pd.read_sql(
    "select subject_code, data, ingested_at from questionnaire_response order by subject_code, ingested_at",
    engine,
)

merged = {}
for subject_code, group in qresp.groupby("subject_code"):
    d = {}
    for blob in group["data"]:
        d.update(json.loads(blob))
    merged[subject_code] = d

questionnaire_wide = pd.DataFrame.from_dict(merged, orient="index").reset_index(names="subject_code")
questionnaire_wide.columns = [
    c if c == "subject_code" else f"q__{c}" for c in questionnaire_wide.columns
]
questionnaire_wide.shape

In [ ]:
# Physiological measurements (SNBB only, keyed by uid): hand_grip + seca (anthropometrics),
# each instrument's own JSON blob — take the reading closest to each subject's scan(s), one row
# per uid per instrument, prefixed by instrument to avoid key collisions.
physio = pd.read_sql(
    "select instrument, uid, data from physiological_measurement where uid is not null",
    engine,
)

physio_wide = None
for instrument, group in physio.groupby("instrument"):
    rows = {}
    for uid, blob in zip(group["uid"], group["data"]):
        rows.setdefault(uid, {}).update(json.loads(blob))
    df = pd.DataFrame.from_dict(rows, orient="index").reset_index(names="subject_key")
    df.columns = [c if c == "subject_key" else f"phys__{instrument}__{c}" for c in df.columns]
    physio_wide = df if physio_wide is None else physio_wide.merge(df, on="subject_key", how="outer")

physio_wide.shape if physio_wide is not None else None

In [ ]:
# CAT12 QC (SIQR grade etc.) — covariate worth checking BAG correlations against directly.
# A session can have QC rows from more than one reprocessing (source='cat12' and 'cat12_v26'),
# which would silently fan out the merge below (confirmed: e.g. session 200109071247 has 2 rows) —
# keep only the most recently ingested row per session.
qc = pd.read_sql(
    "select subject_key, session_id, siqr_pct, siqr_grade, ncr, icr, res_rms_mm, contrast, "
    "gmv_tiv_pct, vol_abs_wmh, vol_rel_wmh, ingested_at from cat12_quality order by ingested_at",
    engine,
)
qc = qc.drop_duplicates(subset=["subject_key", "session_id"], keep="last").drop(columns=["ingested_at"])
qc.columns = [c if c in ("subject_key", "session_id") else f"qc__{c}" for c in qc.columns]
qc.shape

In [ ]:
metadata = demo.merge(questionnaire_wide, on="subject_code", how="left")
metadata = metadata.merge(physio_wide, on="subject_key", how="left")
metadata = metadata.merge(qc, on=["subject_key", "session_id"], how="left")
metadata.shape

## 5. Master table — global BAG + regional BAG + metadata

In [ ]:
master = global_bag_df.merge(regional_bag_df, on=["subject_key", "session_id"], how="inner")
master = master.merge(metadata, on=["subject_key", "session_id"], how="left")

# Region-level analysis (explore()'s ranking, the brain-surface plots) uses the corrected columns —
# raw regional BAG carries the same age-dependent bias the Cole correction exists to remove.
REGION_COLS_RAW = [c for c in master.columns if c.startswith("region_bag_raw__")]
REGION_COLS = [c for c in master.columns if c.startswith("region_bag_corrected__")]
META_COLS = [c for c in master.columns if c not in REGION_COLS and c not in REGION_COLS_RAW and c not in ("subject_key", "session_id", "age_true", "global_bag_raw", "global_bag_corrected")]
print(f"{len(master)} sessions, {len(REGION_COLS)} regions, {len(META_COLS)} metadata columns")
master.head()

In [ ]:
master["days_from_beginning"] = (pd.to_datetime(master["scan_date"]) - pd.to_datetime(master["scan_date"]).min()).dt.days
master["year"] = pd.to_datetime(master["scan_date"]).dt.year

## 6. Explorer

`list_features(pattern)` — browse available metadata column names (e.g. `list_features("q__")` for all questionnaire fields, `list_features("PHQ")` to find one).

`explore(feature, bag_col="global_bag_raw", top_n=15)` — numeric feature: Pearson + Spearman vs. global BAG, scatter plot, and a ranked bar chart of the `top_n` regions whose BAG correlates most strongly (by |Spearman r|). Categorical feature (non-numeric, ≤ 12 distinct values): one-way ANOVA vs. global BAG, boxplot by group, and the `top_n` regions ranked by ANOVA F-stat.

In [ ]:
def list_features(pattern: str = "") -> pd.DataFrame:
    cols = [c for c in META_COLS if pattern.lower() in c.lower()]
    return pd.DataFrame(
        {"column": cols, "dtype": [master[c].dtype for c in cols], "n_non_null": [master[c].notna().sum() for c in cols]}
    ).sort_values("n_non_null", ascending=False).reset_index(drop=True)


def _is_numeric_like(series: pd.Series) -> bool:
    coerced = pd.to_numeric(series, errors="coerce")
    return coerced.notna().sum() >= 0.8 * series.notna().sum() and series.notna().sum() > 0


def _numeric_region_ranking(feature: pd.Series, top_n: int) -> pd.DataFrame:
    rows = []
    for col in REGION_COLS:
        sub = pd.concat([feature, master[col]], axis=1).dropna()
        if len(sub) < 10:
            continue
        r, p = stats.spearmanr(sub.iloc[:, 0], sub.iloc[:, 1])
        rows.append({"region": col.removeprefix("region_bag_corrected__"), "spearman_r": r, "p": p, "n": len(sub)})
    ranking = pd.DataFrame(rows).sort_values("spearman_r", key=lambda s: s.abs(), ascending=False)
    return ranking.head(top_n)


def _categorical_region_ranking(feature: pd.Series, top_n: int) -> pd.DataFrame:
    rows = []
    for col in REGION_COLS:
        sub = pd.concat([feature, master[col]], axis=1).dropna()
        groups_ = [g[col].to_numpy() for _, g in sub.groupby(feature.name) if len(g) >= 3]
        if len(groups_) < 2:
            continue
        f_stat, p = stats.f_oneway(*groups_)
        rows.append({"region": col.removeprefix("region_bag_corrected__"), "F": f_stat, "p": p, "n": len(sub)})
    ranking = pd.DataFrame(rows).sort_values("F", ascending=False)
    return ranking.head(top_n)


def explore(feature_name: str, bag_col: str = "global_bag_raw", top_n: int = 15, specific_groups: list[str] | None = None):
    if feature_name not in master.columns:
        raise KeyError(f"{feature_name!r} not found — try list_features({feature_name.split('__')[0]!r})")

    feature = master[feature_name]
    if feature.notna().sum() == 0:
        print(f"{feature_name} is all-null in the current master table (e.g. an SNBB-only field with the "
              f"master table currently dominated by legacy-cohort sessions) — nothing to correlate.")
        return None

    numeric = _is_numeric_like(feature)

    if numeric:
        feature = pd.to_numeric(feature, errors="coerce")
        sub = pd.concat([feature.rename(feature_name), master[bag_col]], axis=1).dropna()
        if len(sub) < 3:
            print(f"{feature_name} vs {bag_col}: only {len(sub)} non-null overlapping rows — too few to correlate.")
            return None
        pear_r, pear_p = stats.pearsonr(sub[feature_name], sub[bag_col])
        spear_r, spear_p = stats.spearmanr(sub[feature_name], sub[bag_col])
        print(f"{feature_name} vs {bag_col} (n={len(sub)})")
        print(f"  Pearson  r={pear_r:.3f}  p={pear_p:.3g}")
        print(f"  Spearman r={spear_r:.3f}  p={spear_p:.3g}")

        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        axes[0].scatter(sub[feature_name], sub[bag_col], alpha=0.4, s=12)
        axes[0].set_xlabel(feature_name)
        axes[0].set_ylabel(bag_col)
        axes[0].set_title(f"{feature_name} vs {bag_col}")

        ranking = _numeric_region_ranking(feature.rename(feature_name), top_n)
        axes[1].barh(ranking["region"][::-1], ranking["spearman_r"][::-1])
        axes[1].set_xlabel("Spearman r")
        axes[1].set_title(f"top {top_n} regions by |r| with {feature_name}")
        plt.tight_layout()
        plt.show()
        return ranking, sub

    sub = pd.concat([feature.rename(feature_name), master[bag_col]], axis=1).dropna()
    if specific_groups is not None:
        sub = sub[sub[feature_name].isin(specific_groups)]
    n_groups = sub[feature_name].nunique()
    if n_groups > 12:
        print(f"{feature_name} has {n_groups} distinct non-numeric values — not treating as categorical, skipping.")
        return None

    groups_ = [g[bag_col].to_numpy() for _, g in sub.groupby(feature_name) if len(g) >= 3]
    if len(groups_) < 2:
        print(f"{feature_name} vs {bag_col}: only {len(groups_)} group(s) with ≥3 non-null rows — too few to run ANOVA.")
        return None
    f_stat, p = stats.f_oneway(*groups_)
    print(f"{feature_name} vs {bag_col} (n={len(sub)}, {len(groups_)} groups)")
    print(f"  one-way ANOVA F={f_stat:.3f}  p={p:.3g}")

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sub.boxplot(column=bag_col, by=feature_name, ax=axes[0])
    # add stripplot of individual points on top of the boxplot
    for i, group in enumerate(sub[feature_name].unique()):
        y = sub.loc[sub[feature_name] == group, bag_col]
        x = np.random.normal(i + 1, 0.04, size=len(y))  # jitter x positions
        axes[0].scatter(x, y, alpha=0.4, s=12, color='black')
        # also add the n of points in each group as text above the boxplot
        axes[0].text(i + 1, y.max() + 0.02 * (y.max() - y.min()), f"n={len(y)}", ha='center', va='bottom', fontsize=8)
    axes[0].set_title(f"{bag_col} by {feature_name}")
    plt.suptitle("")

    ranking = _categorical_region_ranking(feature.rename(feature_name), top_n)
    axes[1].barh(ranking["region"][::-1], ranking["F"][::-1])
    axes[1].set_xlabel("ANOVA F")
    axes[1].set_title(f"top {top_n} regions by F with {feature_name}")
    plt.tight_layout()
    plt.show()
    return ranking, sub

## 7. Usage

```python
list_features("q__")        # every questionnaire column
list_features("PHQ")        # find a specific instrument
explore("q__PHQ9")          # numeric — correlation + scatter + region ranking
explore("q__Smoking")       # categorical — ANOVA + boxplot + region ranking
explore("qc__siqr_pct")     # sanity check: BAG shouldn't track QC quality
explore("age_true")         # sanity check: raw BAG should be ~decorrelated from age (Cole correction), global_bag_raw won't be
```

In [ ]:
list_features("date").head(20)

In [ ]:
explore("qc__siqr_pct")  # a field with real coverage on the current (mostly-legacy) cohort

In [ ]:
ranking,sub = explore("group_label",bag_col="global_bag_corrected", specific_groups=["H", "MCI", "Stroke"])  # a field with real coverage on the current (mostly-legacy) cohort

In [ ]:
from scipy import stats

group1 = sub[sub["group_label"] == "H"]["global_bag_corrected"]
group2 = sub[sub["group_label"] == "Stroke"]["global_bag_corrected"]
t_stat, p_value = stats.ttest_ind(group1, group2, equal_var=False)  # Welch's t-test
print(f"Welch's t-test: t-statistic = {t_stat:.3f}, p-value = {p_value:.3g}")

In [ ]:
import seaborn as sns
fig, ax = plt.subplots(1,1, figsize=(6, 4))
# ax = sns.kdeplot(data=sub[sub["group_label"].isin(["H", "MCI"])], x="global_bag_corrected", hue="group_label", fill=True, common_norm=False, alpha=0.5, ax=ax)
ax = sns.kdeplot(data=sub, x="global_bag_corrected", hue="group_label", fill=True, common_norm=False, alpha=0.5, ax=ax)
ax.set_title("global_bag_corrected by group_label (H vs MCI)")
ax.set_xlabel("global_bag_corrected")
ax.set_ylabel("Density")




In [ ]:
fig, ax = plt.subplots(1,1, figsize=(6, 4))
sns.boxplot(data=master, x="year", y="global_bag_corrected", ax=ax)
sns.stripplot(data=master, x="year", y="global_bag_corrected", ax=ax, color='black', alpha=0.3, size=3)


## 8. Region-wise results on the brain (yabplot)

Visualizes the most recent `ranking` DataFrame (region + `spearman_r`/`F` column) from
`explore()` on the actual cortical/subcortical surfaces, via the maintainer's own
[yabplot](https://github.com/GalKepler/yabplot) package (`config/local.yaml`'s
`paths.yabplot_repo`, not on PyPI — `uv pip install -e <that path>` to enable this cell).

Region names are `Schaefer2018N400n7Tian2020S2__<region>` (see Phase 2's 2026-08-21/24
notes): cortical regions map onto yabplot's `schaefer400` atlas one-to-one (after
restoring the `7Networks_` prefix yabplot expects). Subcortical regions are Tian
**scale II** (32 regions) — plotted via a custom atlas of per-region meshes
(`config/local.yaml`'s `paths.yabplot_tian_s2_atlas`) whose 32 filenames match our
region names exactly, no yabplot built-in atlas needed.

In [ ]:
import yabplot

value_col = "spearman_r" if "spearman_r" in ranking.columns else "F"
region_values = {col.split("__", 1)[1]: val for col, val in zip(ranking["region"], ranking[value_col])}

cortical_vals = {f"7Networks_{r}": v for r, v in region_values.items() if r.startswith(("LH_", "RH_"))}
subcortical_vals = {r: v for r, v in region_values.items() if not r.startswith(("LH_", "RH_"))}

ax_c = yabplot.plot_cortical(cortical_vals, atlas="schaefer400", cmap="coolwarm")
ax_c.figure.suptitle(f"{value_col} by region — cortical (Schaefer 400x7)")
plt.show()

ax_s = yabplot.plot_subcortical(
    subcortical_vals, custom_atlas_path=str(get_path("yabplot_tian_s2_atlas")), cmap="coolwarm"
)
ax_s.figure.suptitle(f"{value_col} by region — subcortical (Tian scale II)")
plt.show()

In [ ]:
yabplot.get_available_resources()
